#  AegisHealth KBQA — Benchmark Runner (Kaggle)

Notebook này chạy **45 câu benchmark mù** trên các model Ollama lớn hơn để so sánh với kết quả local (qwen2.5:3b).

## Kiến trúc
```
Kaggle Notebook (GPU T4)
  ├── Ollama server    → chạy qwen2.5:7b hoặc 14b
  ├── FastAPI backend  → kết nối Neo4j AuraDB (cloud)
  └── Benchmark script → gọi API, chấm điểm, ghi kết quả
```

##  Checklist trước khi Run All:
- [ ] **Accelerator** → GPU T4 x1 đã bật (Settings → Accelerator → GPU T4 x1)
- [ ] **Internet** → đã bật (Settings → Internet)
- [ ] **Secrets** → đã thêm Neo4j + (tuỳ chọn) Qdrant credentials
  - `NEO4J_URI` — URI của Neo4j AuraDB
  - `NEO4J_USERNAME` — thường là `neo4j`
  - `NEO4J_PASSWORD` — mật khẩu AuraDB
  - `QDRANT_URL` — URL Qdrant Cloud (nếu dùng LightRAG)
  - `QDRANT_API_KEY` — API key Qdrant Cloud
- [ ] Đã chọn model ở Cell 2: `MODEL_NAME = "qwen2.5:7b"` hoặc `"qwen2.5:14b"`

##  Thời gian ước tính
| Model | RAM cần | Thời gian pull | Thời gian benchmark |
|-------|---------|---------------|--------------------|
| qwen2.5:7b | ~4.7 GB | ~5 phút | ~15-25 phút |
| qwen2.5:14b | ~8.7 GB | ~10 phút | ~25-40 phút |

>  Kaggle cấp 16GB RAM + 16GB GPU VRAM (T4). Cả hai model đều chạy được.

In [ ]:
# --- Kiểm tra môi trường ---
import os, platform, subprocess, sys

print("=" * 60)
print(" Environment Check")
print("=" * 60)
print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")

# Check GPU
try:
    gpu_info = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total,memory.free",
                                         "--format=csv,noheader"], text=True).strip()
    print(f"GPU: {gpu_info}")
except Exception:
    print(" No GPU detected — sẽ chạy chậm hơn")

# Check RAM
with open("/proc/meminfo") as f:
    for line in f:
        if line.startswith("MemTotal"):
            ram_gb = int(line.split()[1]) / 1024 / 1024
            print(f"RAM: {ram_gb:.1f} GB")
            break

# Check disk
disk = subprocess.check_output(["df", "-h", "/"], text=True).strip().split("\n")[-1]
print(f"Disk: {disk}")

print()
print(" Environment check done")

In [ ]:
# --- Cấu hình ---
# Thay đổi MODEL_NAME theo nhu cầu:
#   "qwen2.5:3b"  — nhỏ nhất, nhanh nhất (baseline local)
#   "qwen2.5:7b"  — cân bằng (khuyên dùng)
#   "qwen2.5:14b" — mạnh nhất (cần ~9GB RAM)

MODEL_NAME = "qwen2.5:7b"          # ← ĐỔI Ở ĐÂY
BENCHMARK_TIMEOUT = 120            # giây / câu hỏi
API_PORT = 8000
OLLAMA_PORT = 11434
REPO_URL = "https://github.com/NgBaoAnn/kbqa.git"  # ← đổi nếu repo private
REPO_DIR = "/kaggle/working/kbqa"

# ── Lấy credentials từ Kaggle Secrets ────────────────────────────────────────
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

def get_secret(name, default=""):
    try:
        return secrets.get_secret(name)
    except Exception:
        print(f"   Secret '{name}' not found — using default")
        return default

NEO4J_URI      = get_secret("NEO4J_URI")
NEO4J_USERNAME = get_secret("NEO4J_USERNAME", "neo4j")
NEO4J_PASSWORD = get_secret("NEO4J_PASSWORD")
QDRANT_URL     = get_secret("QDRANT_URL")
QDRANT_API_KEY = get_secret("QDRANT_API_KEY")

print(" Config loaded:")
print(f"  Model:     {MODEL_NAME}")
print(f"  Neo4j URI: {NEO4J_URI[:40]}..." if NEO4J_URI else "  Neo4j URI:  NOT SET")
print(f"  Qdrant:    {QDRANT_URL[:40]}..." if QDRANT_URL else "  Qdrant:     NOT SET (LightRAG sẽ không hoạt động)")

In [ ]:
# --- Clone repo (branch develop) ---
import subprocess, os

REPO_BRANCH = "develop"

if os.path.exists(REPO_DIR):
    print(f" Repo đã tồn tại, đang cập nhật branch {REPO_BRANCH}...")
    subprocess.run(["git", "fetch", "origin"], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", REPO_BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "pull", "origin", REPO_BRANCH], cwd=REPO_DIR, check=True)
else:
    print(f" Cloning repo {REPO_URL} (branch={REPO_BRANCH})...")
    subprocess.run(
        ["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, REPO_DIR],
        check=True
    )

# Xác nhận branch
branch = subprocess.check_output(["git", "branch", "--show-current"], cwd=REPO_DIR, text=True).strip()
print(f" Repo ready  •  branch: {branch}")
os.listdir(REPO_DIR)

In [ ]:
# --- Cài Python dependencies ---
import subprocess, sys

print(" Cài packages...")

# Cài editable package (bao gồm ai_engine và backend)
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".", "-q",
     "--no-warn-script-location"],
    cwd=REPO_DIR,
    capture_output=True, text=True
)

if result.returncode != 0:
    print(" Lỗi cài package:")
    print(result.stderr[-2000:])
else:
    print(" Package cài xong")

# Thêm repo vào sys.path để import được
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(f" sys.path: {REPO_DIR} ")

In [ ]:
# --- Cài và khởi động Ollama ---
import subprocess, os, time, requests

OLLAMA_URL = f"http://localhost:{OLLAMA_PORT}"
OLLAMA_MODELS_DIR = "/kaggle/working/ollama_models"

def find_ollama():
    for p in ("/usr/local/bin/ollama", "/usr/bin/ollama", "/bin/ollama"):
        if os.path.exists(p):
            return p
    try:
        return subprocess.check_output(["which", "ollama"], text=True).strip()
    except Exception:
        return None

ollama_bin = find_ollama()

if ollama_bin:
    print(f" Ollama đã có sẵn tại: {ollama_bin}")
else:
    # Cài zstd trước (Ollama install script yêu cầu)
    print(" Cài zstd (dependency của Ollama install script)...")
    r = subprocess.run(
        ["apt-get", "install", "-y", "-q", "zstd"],
        capture_output=True, text=True
    )
    if r.returncode == 0:
        print(" zstd đã cài")
    else:
        print(f" apt-get zstd thất bại (code {r.returncode}): {r.stderr[-200:]}")

    print("  Đang cài Ollama...")
    install_result = subprocess.run(
        "curl -fsSL https://ollama.com/install.sh | sh",
        shell=True, capture_output=True, text=True
    )
    print(install_result.stdout[-1500:])
    if install_result.stderr.strip():
        print("[stderr]:", install_result.stderr[-300:])

    ollama_bin = find_ollama()
    if not ollama_bin:
        print(f" Không tìm thấy ollama binary (returncode={install_result.returncode})")
        raise RuntimeError("Ollama install failed: binary not found")
    print(f" Ollama đã cài thành công tại: {ollama_bin}")

# Khởi động Ollama server ở background
os.makedirs(OLLAMA_MODELS_DIR, exist_ok=True)
print(" Khởi động Ollama server...")
ollama_env = os.environ.copy()
ollama_env["OLLAMA_MODELS"] = OLLAMA_MODELS_DIR
ollama_env["OLLAMA_KEEP_ALIVE"] = "60m"

ollama_log = "/kaggle/working/ollama.log"
with open(ollama_log, "w") as log:
    ollama_proc = subprocess.Popen(
        [ollama_bin, "serve"],
        env=ollama_env,
        stdout=log,
        stderr=log
    )

# Chờ server khởi động (tối đa 40s)
print(" Chờ Ollama server...")
for i in range(40):
    try:
        r = requests.get(f"{OLLAMA_URL}/api/tags", timeout=2)
        if r.status_code == 200:
            print(f" Ollama server ready (sau {i+1}s)")
            break
    except Exception:
        pass
    time.sleep(1)
else:
    print(" Ollama server không khởi động được trong 40s")
    print("\n Ollama log:")
    with open(ollama_log) as f:
        print(f.read()[-2000:])
    raise RuntimeError("Ollama server timeout")

ver = subprocess.check_output([ollama_bin, "--version"], text=True).strip()
print(f"   Version: {ver}")

In [ ]:
# --- Pull model (có thể mất vài phút) ---
import subprocess, time

print(f"  Pulling model: {MODEL_NAME}")
print("  (Có thể mất 5-15 phút tùy model và mạng Kaggle)")
print()

t0 = time.time()
result = subprocess.run(
    [ollama_bin, "pull", MODEL_NAME],
    capture_output=False,  # hiển thị tiến trình
    text=True
)

elapsed = time.time() - t0
if result.returncode != 0:
    print(f"\n Pull model thất bại (code {result.returncode})")
    raise RuntimeError(f"ollama pull {MODEL_NAME} failed")

print(f"\n Model '{MODEL_NAME}' đã sẵn sàng ({elapsed:.0f}s)")

# Kiểm tra model list
models = subprocess.check_output([ollama_bin, "list"], text=True)
print(f"\n Danh sách model:\n{models}")

In [ ]:
# --- Tạo file .env cho backend ---
import os

env_content = f"""# Auto-generated for Kaggle benchmark
# == Neo4j AuraDB ==
NEO4J_URI={NEO4J_URI}
NEO4J_USERNAME={NEO4J_USERNAME}
NEO4J_PASSWORD={NEO4J_PASSWORD}

# == LLM Server (Ollama) ==
LLM_BASE_URL=http://localhost:{OLLAMA_PORT}/v1
LLM_MODEL_NAME={MODEL_NAME}
LLM_TIMEOUT_SECONDS=90

# == Embedding (dùng cùng Ollama server) ==
EMBEDDING_MODEL=nomic-embed-text
EMBEDDING_DIM=768
EMBEDDING_BASE_URL=http://localhost:{OLLAMA_PORT}/v1

# == LightRAG ==
LIGHTRAG_WORKING_DIR={REPO_DIR}/lightrag_data
LIGHTRAG_KG_STORAGE=Neo4JStorage
LIGHTRAG_VECTOR_STORAGE=NanoVectorDBStorage
LIGHTRAG_DOC_STORAGE=JsonKVStorage
DEFAULT_QUERY_MODE=hybrid

# == Qdrant (nếu có) ==
QDRANT_URL={QDRANT_URL}
QDRANT_API_KEY={QDRANT_API_KEY}

# == API ==
API_HOST=0.0.0.0
API_PORT={API_PORT}
LOG_LEVEL=WARNING
"""

env_path = os.path.join(REPO_DIR, ".env")
with open(env_path, "w") as f:
    f.write(env_content)

print(f" .env đã tạo tại: {env_path}")
print("   (credentials được ẩn)")

In [ ]:
# --- Pull embedding model (nomic-embed-text) ---
# nomic-embed-text nhẹ hơn BAAI/bge-m3, phù hợp với Kaggle
# Nếu bạn muốn dùng bge-m3 (dim=1024, phù hợp với Qdrant data), đổi sang:
#   EMBEDDING_MODEL = "nomic-embed-text"  (dim=768)
#   hoặc dùng bge-m3 nhưng cần pull thêm

EMBED_MODEL = "nomic-embed-text"  # dim=768, nhanh
# EMBED_MODEL = "bge-m3"           # dim=1024, khớp Qdrant data nhưng chậm hơn

import subprocess, time
print(f"  Pulling embedding model: {EMBED_MODEL}")

t0 = time.time()
result = subprocess.run([ollama_bin, "pull", EMBED_MODEL], text=True)
elapsed = time.time() - t0

if result.returncode != 0:
    print(f" Pull embedding model thất bại — LightRAG có thể không hoạt động")
else:
    print(f" Embedding model '{EMBED_MODEL}' ready ({elapsed:.0f}s)")

# Cập nhật .env với embedding model đã pull
with open(env_path, "r") as f:
    content = f.read()
content = content.replace("nomic-embed-text", EMBED_MODEL)
with open(env_path, "w") as f:
    f.write(content)

In [ ]:
# --- Khởi động FastAPI backend ---
import subprocess, os, time, requests, sys

backend_log = "/kaggle/working/backend.log"

# backend/app/main.py dùng `from app.config import ...`
# nên cần thêm REPO_DIR/backend vào PYTHONPATH
backend_dir = os.path.join(REPO_DIR, "backend")
pythonpath   = f"{REPO_DIR}:{backend_dir}"

env = os.environ.copy()
env["PYTHONPATH"] = pythonpath
env["NO_PROXY"]   = "localhost,127.0.0.1"
env["no_proxy"]   = "localhost,127.0.0.1"

print(f" Khởi động FastAPI backend (port {API_PORT})...")
print(f"   PYTHONPATH: {pythonpath}")

with open(backend_log, "w") as log_file:
    backend_proc = subprocess.Popen(
        [
            sys.executable, "-m", "uvicorn",
            "backend.app.main:app",
            "--host", "0.0.0.0",
            "--port", str(API_PORT),
            "--log-level", "warning",
        ],
        cwd=REPO_DIR,
        env=env,
        stdout=log_file,
        stderr=log_file
    )

# LightRAG init có thể mất 2-3 phút trên Kaggle
API_URL = f"http://localhost:{API_PORT}"
MAX_WAIT = 240  # 4 phút
print(f" Chờ backend tại {API_URL} (tối đa {MAX_WAIT}s)...")
print("   (LightRAG init có thể chậm — theo dõi tiến trình 30s/lần)")

for i in range(MAX_WAIT):
    # Kiểm tra process còn sống không
    if backend_proc.poll() is not None:
        exit_code = backend_proc.poll()
        print(f"\n Backend đã crash (exit code={exit_code})")
        print("\n Backend log:")
        with open(backend_log) as f:
            print(f.read()[-4000:])
        raise RuntimeError(f"Backend crashed with exit code {exit_code}")

    # Thử kết nối
    try:
        r = requests.get(f"{API_URL}/health", timeout=3)
        if r.status_code == 200:
            print(f"\n Backend ready (sau {i+1}s)")
            break
    except Exception:
        pass

    # Log tiến trình mỗi 30s
    if (i + 1) % 30 == 0:
        print(f"   [{i+1}s] Vẫn đang chờ... (backend process pid={backend_proc.pid})")
        # In 3 dòng log cuối để biết đang làm gì
        try:
            with open(backend_log) as f:
                lines = f.read().strip().split("\n")
                for line in lines[-3:]:
                    print(f"      {line}")
        except Exception:
            pass

    time.sleep(1)
else:
    print(f"\n Backend không khởi động được trong {MAX_WAIT}s")
    print("\n Backend log (4000 ký tự cuối):")
    with open(backend_log) as f:
        print(f.read()[-4000:])
    raise RuntimeError("Backend startup timeout")

# Kiểm tra health
r = requests.get(f"{API_URL}/health")
print(f" Health check: {r.json()}")

In [ ]:
# --- Smoke test — thử 1 câu hỏi ---
import requests, json

test_q = "viêm gan B có những triệu chứng gì?"
print(f" Smoke test: '{test_q}'")
print()

r = requests.post(
    f"{API_URL}/api/v1/query",
    json={"question": test_q},
    timeout=120
)

if r.status_code == 200:
    data = r.json()
    print(f" Status: {r.status_code}")
    print(f"   Engine:     {data.get('engine', 'N/A')}")
    print(f"   Query type: {data.get('query_type', 'N/A')}")
    print(f"   Answer preview: {str(data.get('answer', ''))[:200]}...")
else:
    print(f" Status: {r.status_code}")
    print(r.text[:500])

In [ ]:
# --- Chạy Benchmark ---
# Đây là cell chính — chạy tất cả 45 câu và ghi kết quả
import subprocess, sys, os, time
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
out_file  = f"/kaggle/working/benchmark_{MODEL_NAME.replace(':', '_')}_{timestamp}.md"

benchmark_script = os.path.join(REPO_DIR, "ai_engine", "eval", "run_benchmark.py")

print(" Bắt đầu benchmark...")
print(f"   Model:   {MODEL_NAME}")
print(f"   API URL: {API_URL}")
print(f"   Timeout: {BENCHMARK_TIMEOUT}s/câu")
print(f"   Output:  {out_file}")
print()

t0 = time.time()
result = subprocess.run(
    [
        sys.executable,
        benchmark_script,
        "--url", API_URL,
        "--out", out_file,
        "--timeout", str(BENCHMARK_TIMEOUT),
    ],
    cwd=REPO_DIR,
    env={**os.environ,
         "PYTHONPATH": REPO_DIR,
         "NO_PROXY": "localhost,127.0.0.1",
         "no_proxy": "localhost,127.0.0.1"},
    capture_output=False,
    text=True
)

total_time = time.time() - t0
print()
print("=" * 60)
if result.returncode == 0:
    print(f" Benchmark hoàn thành trong {total_time:.0f}s ({total_time/60:.1f} phút)")
    print(f" Kết quả: {out_file}")
else:
    print(f" Benchmark thất bại (code {result.returncode})")

In [ ]:
# --- Hiển thị kết quả ---
from IPython.display import Markdown, display
import os

if os.path.exists(out_file):
    with open(out_file, "r") as f:
        content = f.read()
    print(f" File: {out_file}")
    print(f" Size: {len(content):,} bytes")
    print()
    display(Markdown(content))
else:
    print(f" File không tồn tại: {out_file}")

In [ ]:
# --- So sánh với baseline (qwen2.5:3b) ---
# Paste kết quả baseline từ file local vào đây để so sánh
# Hoặc upload file baseline lên Kaggle để đọc

import re

def parse_summary(filepath):
    """Trích xuất số liệu từ benchmark report."""
    if not os.path.exists(filepath):
        return None
    
    with open(filepath) as f:
        content = f.read()
    
    # Tìm dòng tổng kết
    stats = {}
    
    # PASS/FAIL counts
    m = re.search(r'PASS.*?(\d+).*?FAIL.*?(\d+)', content)
    if not m:
        m = re.search(r'(\d+).*?pass.*?(\d+).*?fail', content, re.IGNORECASE)
    
    # Engine distribution
    cypher_hits = len(re.findall(r'cypher_direct', content))
    lightrag_hits = len(re.findall(r'lightrag', content))
    
    # Pass rate từ table
    pass_count = len(re.findall(r'\| ', content))
    fail_count = len(re.findall(r'\| ', content))
    total = pass_count + fail_count
    
    return {
        "pass": pass_count,
        "fail": fail_count,
        "total": total,
        "pass_rate": f"{pass_count/total*100:.1f}%" if total > 0 else "N/A",
    }

print(f" Kết quả model: {MODEL_NAME}")
stats = parse_summary(out_file)
if stats:
    print(f"   PASS: {stats['pass']}/{stats['total']} ({stats['pass_rate']})")
    print(f"   FAIL: {stats['fail']}/{stats['total']}")
else:
    print("    Không parse được kết quả")

print()
print(" Tip: Download file kết quả từ Output panel bên phải để so sánh với baseline local")

In [ ]:
# --- [TÙY CHỌN] Chạy thêm model khác để so sánh ---
# Uncomment và chạy cell này để benchmark thêm model thứ 2

# MODEL_2 = "qwen2.5:14b"
# out_file_2 = f"/kaggle/working/benchmark_{MODEL_2.replace(':', '_')}_{timestamp}.md"
# 
# print(f"  Pulling model 2: {MODEL_2}")
# subprocess.run([ollama_bin, "pull", MODEL_2])
# 
# # Cập nhật .env với model mới
# with open(env_path, "r") as f:
#     env_content = f.read()
# env_content = env_content.replace(f"LLM_MODEL_NAME={MODEL_NAME}", f"LLM_MODEL_NAME={MODEL_2}")
# with open(env_path, "w") as f:
#     f.write(env_content)
# 
# # Restart backend với model mới (cần reload env)
# backend_proc.terminate()
# time.sleep(3)
# # ... khởi động lại (copy Cell 9)
# 
# # Chạy benchmark
# subprocess.run([
#     sys.executable, benchmark_script,
#     "--url", API_URL, "--out", out_file_2, "--timeout", str(BENCHMARK_TIMEOUT)
# ], cwd=REPO_DIR)
# 
# # So sánh
# s1 = parse_summary(out_file)
# s2 = parse_summary(out_file_2)
# print(f"{MODEL_NAME:20s}: PASS={s1['pass']}/{s1['total']} ({s1['pass_rate']})")
# print(f"{MODEL_2:20s}: PASS={s2['pass']}/{s2['total']} ({s2['pass_rate']})")

print("Cell này để chạy model thứ 2. Uncomment code để dùng.")

In [ ]:
# --- Dọn dẹp ---
# Chạy cell này KHI ĐÃ XONG để dừng các process nền

import subprocess

print(" Dừng backend và Ollama...")

try:
    backend_proc.terminate()
    print("   Backend stopped")
except Exception as e:
    print(f"   Backend: {e}")

try:
    ollama_proc.terminate()
    print("   Ollama stopped")
except Exception as e:
    print(f"   Ollama: {e}")

print()
print(" Files đã tạo:")
import glob
for f in glob.glob("/kaggle/working/benchmark_*.md"):
    size = os.path.getsize(f)
    print(f"  {f} ({size:,} bytes)")

print()
print(" Done! Tải file kết quả từ Output panel.")